# S2 · Do we benefit from different forecasting models by group?

We profile the Stage 1 partition at **K=3** and test whether group membership improves model choice for **30 daily balances per customer**. We examine activity, sign, recurrence and change over time. In the pilot, we compare smoothing, trend, calendar and pretrained models on the same historical windows.

We use the prepared 500-account sample through `2022-12-31` from [S0](S0__preprocessing.ipynb) and the two Chronos checkpoints described below. We rebuild the Stage 1 partition for the profiles and refit it at eight earlier forecast origins for the pilot. These fits take a few minutes.

## 1 · Reproduction of the selected partition

We use the same six feature families, scaling, dissimilarities and spectral clustering as Stage 1. We fit K=3 directly and label groups using the same medoid ordering. We report group counts and negative silhouette shares to assess size and boundary assignments. A negative silhouette means the account is closer on average to another group in the fitted dissimilarity space.

In [1]:
import sys
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl
import yaml
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.metrics import silhouette_samples

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/bank_balance_forecasting").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))
pio.renderers.default = "plotly_mimetype"
pl.Config.set_tbl_rows(12)
pl.Config.set_tbl_cols(6)
pl.Config.set_fmt_str_lengths(56)

DATASET = "medium"
CUTOFF = date(2022, 12, 31)
HOLDOUT_START = date(2022, 1, 1)
source = ROOT / "data" / "prepared" / DATASET / str(CUTOFF)
if not (source / "prepared_balances.parquet").exists():
    raise FileNotFoundError(f"We could not find Stage 0 prepared balances for {DATASET} through {CUTOFF}.")

settings = yaml.safe_load((ROOT / "parameters.yml").read_text())
parameters = {**settings["stage1"], "as_of_date": str(CUTOFF)}
balances = pl.read_parquet(source / "prepared_balances.parquet")
calendar = pl.read_parquet(source / "calendar_profiles.parquet")
events = pl.read_parquet(source / "event_flags.parquet")
assert balances["date"].max() == CUTOFF
assert events.is_empty() or events["date"].max() <= CUTOFF

COLORS = {"G01": "#227C8D", "G02": "#BE5D36", "G03": "#6658A4"}


def show(fig, height=420):
    """Show a responsive study figure."""
    fig.update_layout(
        autosize=True,
        height=height,
        template="plotly_white",
        font_size=12,
        title_font_size=15,
        margin=dict(l=62, r=25, t=64, b=66),
        legend=dict(orientation="h", x=0, y=-0.22),
    )
    fig.show(config={"responsive": True, "displaylogo": False})

In [2]:
from bank_balance_forecasting.clustering.features import (
    extract_activity,
    extract_balance_state,
    extract_dynamics,
    extract_regime_features,
    feature_label,
)
from bank_balance_forecasting.clustering.graphs import (
    extract_hvg_features,
    extract_nvg_features,
)
from bank_balance_forecasting.clustering.representation import (
    build_representation,
    distance_array,
)
from bank_balance_forecasting.clustering.spectral import spectral_partition

series = balances.lazy()
features, matrix, registry, distances = build_representation(
    extract_activity(series, parameters).lazy(),
    extract_balance_state(series, parameters).lazy(),
    extract_dynamics(series, calendar.lazy(), parameters).lazy(),
    extract_regime_features(series, events.lazy(), parameters).lazy(),
    extract_nvg_features(series, parameters).lazy(),
    extract_hvg_features(series, parameters).lazy(),
    parameters,
)
eligible = matrix.filter("eligible").sort("account_id")
distance = distance_array(distances)
fit = spectral_partition(distance, 3, parameters)
labels = fit["labels"]
assignments = eligible.select("account_id").with_columns(
    pl.Series("group", [f"G{label + 1:02d}" for label in labels]),
    pl.Series("silhouette", silhouette_samples(distance, labels, metric="precomputed")),
)
representatives = dict(zip(COLORS, eligible["account_id"].gather(fit["medoids"]).to_list()))
print(f"Accounts in the three groups: {assignments.height}/{matrix.height}")
display(
    assignments.group_by("group")
    .agg(
        pl.len().alias("Accounts"),
        pl.col("silhouette").mean().round(3).alias("Mean silhouette"),
        ((pl.col("silhouette") < 0).mean() * 100).round(1).alias("Negative silhouette (%)"),
    )
    .sort("group")
)

Accounts in the three groups: 500/500


group,Accounts,Mean silhouette,Negative silhouette (%)
str,u32,f64,f64
"""G01""",151,0.167,3.3
"""G02""",163,0.071,28.8
"""G03""",186,0.234,0.5


## 2 · Features that distinguish the fitted groups

For each retained descriptor, we compare the median within a group with the median among the other grouped accounts. We report the largest absolute difference within each feature family and preserve its sign on the fitted 0 to 1 scale. A positive difference means the descriptor tends to be higher in the group. We use these differences to describe membership. Family weights and correlations also affect the fitted distances.

In [3]:
scaled = eligible.join(assignments.select("account_id", "group"), on="account_id")
signatures = []
for group in COLORS:
    inside = scaled.filter(pl.col("group") == group)
    outside = scaled.filter(pl.col("group") != group)
    for family in registry["weights"]:
        columns = [name for name in registry["features"] if name.startswith(f"{family}__")]
        gaps = []
        for name in columns:
            a = inside[name].drop_nans().median()
            b = outside[name].drop_nans().median()
            if a is not None and b is not None:
                gaps.append((name, a - b))
        if gaps:
            name, gap = max(gaps, key=lambda pair: abs(pair[1]))
            signatures.append((group, family.upper(), feature_label(name), round(gap, 3)))
with pl.Config(tbl_rows=24, tbl_width_chars=115):
    display(
        pl.DataFrame(
            signatures, schema=["Group", "Family", "Descriptor", "Median gap"], orient="row"
        )
    )

Group,Family,Descriptor,Median gap
str,str,str,f64
"""G01""","""ACTIVITY""","""Typical quiet-period length""",0.5
"""G01""","""STATE""","""Frequency of crossing zero""",-0.387
"""G01""","""DYNAMICS""","""Relative Tuesday activity""",0.356
"""G01""","""REGIME""","""Time since latest level-shift candidate""",-0.364
"""G01""","""NVG""","""nvg: changes degree entropy median""",-0.42
"""G01""","""HVG""","""hvg: changes max degree share median""",-0.567
"""G02""","""ACTIVITY""","""Longest quiet period relative to history""",0.175
"""G02""","""STATE""","""Time with a negative balance""",0.706
"""G02""","""DYNAMICS""","""Regularity of monthly largest inflows""",-0.298


## 3 · Forecast-relevant behaviour in familiar units

We use one value per customer for each measure and report the median and interquartile range within each group. We examine movement frequency to see how often a flat forecast would encounter a change, and flat-run lengths to estimate how long recent levels tend to persist. Negative-balance shares and sign crossings describe the range that models must handle. We use lag-1 and lag-7 autocorrelations of daily changes to check for immediate reversal and weekly persistence.

We use the figure menu to compare the distributions of the four main behaviours. Each box covers the middle half of accounts. We interpret overlap as evidence that a group-level description gives limited guidance for some individual accounts.

In [4]:
account_profiles = features.join(assignments.select("account_id", "group"), on="account_id")
metrics = {
    "Movement frequency (%)": ("activity__movement_rate", 100),
    "Typical flat run (days)": ("activity__median_flat_run", 1),
    "Negative balances (%)": ("state__negative_share", 100),
    "Sign crossings (%)": ("state__zero_crossing_rate", 100),
    "Lag-1 change autocorrelation": ("dynamics__acf_change_1", 1),
    "Lag-7 change autocorrelation": ("dynamics__acf_change_7", 1),
}
summary = []
for label, (column, scale) in metrics.items():
    row = {"Measure": label}
    for group in COLORS:
        values = (
            account_profiles.filter(pl.col("group") == group)[column].drop_nans().drop_nulls()
            * scale
        )
        if values.is_empty():
            row[group] = "Unavailable"
        else:
            q25, q75 = [values.quantile(q, interpolation="linear") for q in (0.25, 0.75)]
            row[group] = f"{values.median():.2f} [{q25:.2f}, {q75:.2f}]"
    summary.append(row)
display(pl.DataFrame(summary))

fig = go.Figure()
options = list(metrics)[:4]
for i, label in enumerate(options):
    column, scale = metrics[label]
    for group, color in COLORS.items():
        values = (
            account_profiles.filter(pl.col("group") == group)[column].drop_nans().drop_nulls()
            * scale
        )
        fig.add_box(
            x=values.to_list(),
            name=group,
            marker_color=color,
            boxpoints=False,
            visible=(i == 0),
            orientation="h",
        )
buttons = [
    dict(
        label=label,
        method="update",
        args=[
            {"visible": [j // len(COLORS) == i for j in range(len(fig.data))]},
            {"xaxis.title.text": label, "xaxis.autorange": True},
        ],
    )
    for i, label in enumerate(options)
]
fig.update_layout(
    title="How account behaviour varies within each group",
    xaxis_title=options[0],
    showlegend=False,
    updatemenus=[dict(buttons=buttons, x=1, y=1.02, xanchor="right", yanchor="bottom")],
)
show(fig)

Measure,G01,G02,G03
str,str,str,str
"""Movement frequency (%)""","""37.08 [32.18, 41.73]""","""44.06 [35.69, 50.23]""","""55.48 [50.19, 59.51]"""
"""Typical flat run (days)""","""2.00 [2.00, 2.00]""","""1.00 [1.00, 2.00]""","""1.00 [1.00, 1.00]"""
"""Negative balances (%)""","""0.09 [0.00, 5.89]""","""54.79 [38.03, 70.05]""","""1.74 [0.00, 11.44]"""
"""Sign crossings (%)""","""0.09 [0.00, 1.33]""","""6.58 [4.89, 8.18]""","""0.91 [0.00, 3.11]"""
"""Lag-1 change autocorrelation""","""-0.02 [-0.06, -0.00]""","""-0.07 [-0.12, -0.02]""","""-0.03 [-0.08, -0.00]"""
"""Lag-7 change autocorrelation""","""-0.01 [-0.03, 0.00]""","""-0.02 [-0.05, 0.00]""","""-0.01 [-0.04, 0.00]"""


## 4 · Evidence on calendar timing

**Weekday activity.** For each account and weekday, we divide movement days by adjacent observed pairs and plot the median rate across accounts in each group. This shows when balances are more likely to move and motivates weekday predictors for daily changes. We examine signed-change autocorrelations separately to assess whether the previous week's balance path might repeat.

**Monthly inflow timing.** We identify the largest positive daily balance change in each complete month with a valid change on every day. Using 2020–2021, we estimate each account's preferred day by maximizing how often the peak falls within two calendar days of it. We fix that day and count matches in 2022. We report median account-level match rates and the number of accounts with at least six usable months in each period. This checks whether an account-specific calendar signal recurs. Because we fit these groups using history through 2022, we interpret the timing comparison descriptively. We rebuild groups at each forecast origin in Stage 3.

In [5]:
weekday = (
    calendar.group_by("account_id", "weekday")
    .agg((pl.col("movement_days").sum() / pl.col("eligible_pairs").sum()).alias("movement_rate"))
    .filter(pl.col("movement_rate").is_finite())
    .join(assignments.select("account_id", "group"), on="account_id")
    .group_by("group", "weekday")
    .agg((100 * pl.col("movement_rate").median()).alias("Movement days (%)"))
    .sort("group", "weekday")
    .with_columns(
        pl.col("weekday")
        .replace_strict(dict(enumerate(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 1)))
        .alias("Weekday")
    )
)
fig = px.line(
    weekday.to_pandas(),
    x="Weekday",
    y="Movement days (%)",
    color="group",
    markers=True,
    color_discrete_map=COLORS,
    title="Movement frequency by weekday",
)
show(fig)

monthly_peaks = (
    balances.sort("account_id", "date")
    .with_columns(
        pl.col("balance").diff().over("account_id").alias("change"),
        pl.col("date").dt.month_start().alias("month"),
    )
    .group_by("account_id", "month")
    .agg(
        pl.col("change").count().alias("valid_days"),
        pl.col("change").max().alias("peak"),
        pl.col("date")
        .sort_by("change", descending=True, nulls_last=True)
        .first()
        .dt.day()
        .alias("day"),
    )
    .filter(
        (pl.col("valid_days") == pl.col("month").dt.month_end().dt.day())
        & (pl.col("peak") > parameters["tolerance"])
    )
    .sort("account_id", "month")
)

rows = []
for (account,), months in monthly_peaks.group_by("account_id", maintain_order=True):
    earlier = months.filter(pl.col("month") < HOLDOUT_START)["day"].to_numpy()
    later = months.filter(pl.col("month") >= HOLDOUT_START)["day"].to_numpy()
    if min(len(earlier), len(later)) < 6:
        continue
    days = np.arange(1, 32)
    fits = (np.abs(earlier[:, None] - days) <= 2).mean(axis=0)
    best = days[np.argmax(fits)]
    rows.append((account, float(fits.max()), float((np.abs(later - best) <= 2).mean())))
timing = pl.DataFrame(
    rows, schema={"account_id": pl.String, "early": pl.Float64, "later": pl.Float64}, orient="row"
).join(assignments.select("account_id", "group"), on="account_id")
timing_summary = timing.group_by("group").agg(
    pl.len().alias("Accounts tested"),
    (100 * pl.col("early").median()).round(1).alias("2020–2021 fit (%)"),
    (100 * pl.col("later").median()).round(1).alias("2022 match (%)"),
)
display(
    assignments.group_by("group")
    .len(name="Group accounts")
    .join(timing_summary, on="group", how="left")
    .sort("group")
)

group,Group accounts,Accounts tested,2020–2021 fit (%),2022 match (%)
str,u32,u32,f64,f64
"""G01""",151,150,78.3,66.7
"""G02""",163,161,56.5,40.0
"""G03""",186,183,73.9,66.7


## 5 · Representative group histories

We choose the medoid for each group, the account with the smallest total fitted dissimilarity to the other members. We show its last 120 calendar days in euros. Separate vertical scales preserve detail for accounts of different sizes. We examine flat runs, zero crossings and the frequency of changes. The group distributions show how widely those patterns occur beyond the examples.

In [6]:
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=[f"{group} medoid" for group in COLORS],
)
start = CUTOFF - timedelta(days=119)
for row, (group, account) in enumerate(representatives.items(), 1):
    history = balances.filter((pl.col("account_id") == account) & (pl.col("date") >= start)).sort(
        "date"
    )
    fig.add_trace(
        go.Scatter(
            x=history["date"].to_list(),
            y=history["balance"].to_list(),
            mode="lines",
            line_color=COLORS[group],
            showlegend=False,
        ),
        row=row,
        col=1,
    )
    fig.add_hline(y=0, line_dash="dot", line_color="#adb5bd", row=row, col=1)
    fig.update_yaxes(title_text="EUR", row=row, col=1)
fig.update_layout(title="Representative balances over the last 120 days")
show(fig, 580)

## 6 · Design of the 30-day forecasting experiment

We fit every candidate separately to each account using its history at the forecast origin. We rebuild the Stage 1 groups from balances through that date and calculate event descriptors from follow-up available by then. We map each partition by its training profiles. **G02** has the highest median share of negative balances. Of the other two, **G01** has lower movement frequency and **G03** has higher movement frequency. The table shows whether these distinctions persist across origins.

We select models on the first five origins from April through August 2022. We use the next three origins, from September through November, to evaluate those choices on 30-day windows in October, November and December. Each account-origin must have a current balance, at least 91 observations available at the origin and all 30 target days. Short histories and missing target days would reduce the scored coverage shown below. This is a retrospective pilot. We formed the model hypotheses using the descriptive full-sample analysis above. A prospective or untouched evaluation would give stronger evidence.

We use the following labels in the pilot comparisons.

| Scorecard label | Forecasting method |
|---|---|
| `Last balance` | Persistence. We repeat the most recent observed balance for all 30 days. |
| `SMA 7` | 7-day simple moving average. We forecast one constant level from the last seven daily balances. |
| `SMA 14` | 14-day simple moving average. We forecast one constant level from the last 14 daily balances. |
| `SMA 28` | 28-day simple moving average. We forecast one constant level from the last 28 daily balances. |
| `Seasonal naive 7` | Weekly seasonal persistence. We cycle through the last seven daily balances. |
| `Local level` | Simple exponential smoothing. We fit a constant forecast level using up to 365 days and a fixed level gain of 0.20. |
| `Damped Holt` | Holt's damped trend method. We use up to 365 days, a level gain of 0.20, a trend gain of 0.05 and damping of 0.90. |
| `Drift 90` | Linear drift. We extrapolate the slope between the origin balance and the balance 90 days earlier. |
| `AR(1) changes` | First-order autoregression on daily balance changes. We forecast changes and accumulate them from the last level. |
| `Weekday changes` | Ridge regression on daily changes with weekday indicators. We accumulate the predicted changes. |
| `Calendar changes` | Ridge regression on daily changes with weekday indicators and an account-specific monthly inflow window when available. We accumulate the predicted changes. |
| `Two-part calendar` | Logistic regression for movement probability and ridge regression for signed size on movement days, using calendar features. We multiply the predictions and accumulate the expected changes. |
| `Chronos-Bolt Tiny` | Zero-shot median forecast from Amazon's pretrained `amazon/chronos-bolt-tiny` checkpoint. We use up to 512 daily history positions per account. |
| `Chronos-2 Small` | Zero-shot median forecast from the pretrained `autogluon/chronos-2-small` checkpoint. We use up to 512 daily history positions per account. |

We fix the level and trend smoothing gains and calendar penalties for this pilot. The change models accumulate predicted changes from the current observed balance. The 7-day seasonal baseline repeats balance levels, so it tests a stronger form of recurrence than a weekday movement rate. [Benchmark methods](https://otexts.com/fpp3/toolbox.html), [time series cross-validation](https://otexts.com/fpp3/toolbox.html#time-series-cross-validation), [exponential smoothing](https://otexts.com/fpp3/expsmooth.html), [dynamic regression](https://otexts.com/fpp3/dynamic-regression.html).

**Foundation benchmark.** We use the `chronos` project extra and fetch the two public checkpoints with the commands below. For a classical-only comparison, we set `RUN_CHRONOS = False`. The forecast calls load cached weights and run locally. Each model receives one account history at a time with batching for speed. [Chronos-Bolt Tiny](https://huggingface.co/amazon/chronos-bolt-tiny), [Chronos-2 Small](https://huggingface.co/autogluon/chronos-2-small).

```text
hf download amazon/chronos-bolt-tiny config.json model.safetensors --revision a0e552de83495b5c28c14c71c374f3e33280b340
hf download autogluon/chronos-2-small config.json model.safetensors --revision ddec01313e50b6bc58ebaa92ede81bc24a3d9f9a
```

In [10]:
from bank_balance_forecasting.forecast_pilot import (
    FOUNDATION_MODELS,
    add_foundation_scores,
    compare_policies,
    predict_chronos,
    random_route_control,
    run_pilot,
)

RUN_CHRONOS = True
classical, keys, contexts, actuals, origin_profiles, coverage = run_pilot(balances, settings)
scores = classical
if RUN_CHRONOS:
    for model in FOUNDATION_MODELS:
        forecasts = predict_chronos(contexts, model)
        scores = add_foundation_scores(scores, keys, forecasts, actuals, model)
print(f"{scores['model'].n_unique()} candidates · {keys.height} account-origins")
display(coverage)

group_ranges = (
    origin_profiles.group_by("group")
    .agg(
        pl.col("accounts").min().alias("Smallest group size"),
        pl.col("accounts").max().alias("Largest group size"),
        (100 * pl.col("movement").min()).round(1).alias("Lowest movement median (%)"),
        (100 * pl.col("movement").max()).round(1).alias("Highest movement median (%)"),
        (100 * pl.col("negative").median()).round(1).alias("Typical negative share (%)"),
    )
    .sort("group")
)
display(group_ranges)

14 candidates · 4000 account-origins


origin,split,total,scored
date,str,i64,i64
2022-04-30,"""Selection""",500,500
2022-05-31,"""Selection""",500,500
2022-06-30,"""Selection""",500,500
2022-07-31,"""Selection""",500,500
2022-08-31,"""Selection""",500,500
2022-09-30,"""Evaluation""",500,500
2022-10-31,"""Evaluation""",500,500
2022-11-30,"""Evaluation""",500,500


group,Smallest group size,Largest group size,Lowest movement median (%),Highest movement median (%),Typical negative share (%)
str,u32,u32,f64,f64,f64
"""G01""",143,154,36.4,37.3,0.1
"""G02""",163,173,42.8,44.1,58.3
"""G03""",179,190,54.6,55.4,1.4


## 7 · Trend and calendar signals in 30-day forecasts

**Trend persistence.** We fit a line to weekly median balances in each of the last two 13-week blocks. We count the trend as persistent when both slopes have the same sign and each implies a 12-week change larger than a quarter of the balance interquartile range across both blocks. We use the persistence rate to motivate testing trend models and use forecast errors to assess whether extrapolating helps.

**Forecast skill.** We average each account's absolute euro error across its 30 target days, then average those results within each group. The heatmap shows improvement over last balance on the three evaluation origins. Positive values mean lower error. We compare trend with smoothing and weekday models with inflow-window models to assess which structure helps. We use these comparisons to diagnose errors on the evaluation period. We choose models for the policy comparison on the five earlier origins.

In [11]:
trend_summary = (
    keys.group_by("group")
    .agg(
        pl.col("trend_repeats").count().alias("Complete account-origins"),
        (100 * pl.col("trend_repeats").mean()).round(1).alias("Persistent trend (%)"),
    )
    .sort("group")
)
display(trend_summary)

by_model = scores.group_by("split", "group", "model").agg(pl.col("mae").mean())
evaluation = by_model.filter((pl.col("split") == "Evaluation") & (pl.col("group") != "Review"))
naive = evaluation.filter(pl.col("model") == "Last balance").select(
    "group", pl.col("mae").alias("last_mae")
)
skill = evaluation.join(naive, on="group").with_columns(
    (100 * (1 - pl.col("mae") / pl.col("last_mae"))).alias("skill")
)
heatmap = skill.pivot(on="group", index="model", values="skill").sort("model", descending=True)
values = heatmap.select(list(COLORS)).to_numpy()
fig = go.Figure(
    go.Heatmap(
        x=list(COLORS),
        y=heatmap["model"].to_list(),
        z=values.tolist(),
        text=[[f"{value:+.1f}%" for value in row] for row in values],
        texttemplate="%{text}",
        colorscale="RdBu",
        zmid=0,
        showscale=False,
        hovertemplate="%{x} · %{y}<br>%{z:+.1f}% improvement<extra></extra>",
    )
)
fig.update_layout(title="30-day MAE improvement over last balance")
fig.update_xaxes(side="top")
fig.update_yaxes(automargin=True)
show(fig, 540)

with pl.Config(tbl_rows=18):
    display(
        evaluation.pivot(on="group", index="model", values="mae")
        .sort("model")
        .select("model", *COLORS)
        .with_columns(pl.col(list(COLORS)).round(2))
    )

group,Complete account-origins,Persistent trend (%)
str,u32,f64
"""G01""",1175,36.5
"""G02""",1341,32.5
"""G03""",1484,32.5


model,G01,G02,G03
str,f64,f64,f64
"""AR(1) changes""",1130.94,393.47,4427.14
"""Calendar changes""",1153.33,415.29,4554.89
"""Chronos-2 Small""",1124.49,274.84,4133.96
"""Chronos-Bolt Tiny""",1142.61,288.71,4560.24
"""Damped Holt""",1198.2,359.35,4471.52
"""Drift 90""",1395.82,449.53,5184.76
"""Last balance""",1133.95,393.46,4168.2
"""Local level""",1345.42,348.75,4427.68
"""SMA 14""",1622.45,359.03,4839.84


## 8 · Value of routing forecasts by group

We select the lowest-MAE candidate using the first five origins. We select one model family across accounts for the common policy and a family within each group for the routed policy, with the common choice used for review cases. We use the same candidate menu and fit each selected model separately to each account. We keep the selection rule fixed across the three evaluation origins.

We report paired MAE differences in euros for the same account-origins, overall and by group and date. A negative delta favors routing. We use the median delta and the share of accounts that benefit to assess how widely routing helps. We repeat the selection rule with 50 random groupings of matching sizes, keeping a fixed random account rank across dates. This control estimates how much improvement arbitrary extra model choices can produce in this sample.

In [12]:
common_model, chosen, paired = compare_policies(scores)
print(f"Common model selected in the earlier windows: {common_model}")
display(chosen)

policy_metrics = [
    pl.len().alias("Account-origins"),
    pl.col("common_mae").mean().round(2).alias("Common MAE (€)"),
    pl.col("routed_mae").mean().round(2).alias("Routed MAE (€)"),
    pl.col("delta").mean().round(2).alias("Mean delta (€)"),
    pl.col("delta").median().round(2).alias("Median delta (€)"),
    (100 * (pl.col("delta") < -1e-8).mean()).round(1).alias("Helped (%)"),
]
with pl.Config(tbl_cols=8):
    display(paired.select(policy_metrics))
    display(paired.group_by("group").agg(policy_metrics).sort("group"))
    display(paired.group_by("origin").agg(policy_metrics).sort("origin"))

random = random_route_control(scores)
q05, median, q95 = (random["delta"].quantile(q, interpolation="linear") for q in (0.05, 0.50, 0.95))
print(
    f"Random routing delta in euros · median {median:+.2f} · 5th to 95th percentile {q05:+.2f} to {q95:+.2f}"
)
if RUN_CHRONOS:
    simple_common, simple_chosen, simple_paired = compare_policies(classical)
    print(f"Classical-only common model: {simple_common}")
    display(simple_chosen)
    display(simple_paired.select(policy_metrics))

Common model selected in the earlier windows: Chronos-2 Small


group,routed_model
str,str
"""G01""","""Chronos-2 Small"""
"""G02""","""Chronos-2 Small"""
"""G03""","""SMA 28"""


Account-origins,Common MAE (€),Routed MAE (€),Mean delta (€),Median delta (€),Helped (%)
u32,f64,f64,f64,f64,f64
1500,1923.03,2259.39,336.36,0.0,10.2


group,Account-origins,Common MAE (€),Routed MAE (€),Mean delta (€),Median delta (€),Helped (%)
str,u32,f64,f64,f64,f64,f64
"""G01""",448,1124.49,1124.49,0.0,0.0,0.0
"""G02""",510,274.84,274.84,0.0,0.0,0.0
"""G03""",542,4133.96,5064.84,930.88,87.47,28.2


origin,Account-origins,Common MAE (€),Routed MAE (€),Mean delta (€),Median delta (€),Helped (%)
date,u32,f64,f64,f64,f64,f64
2022-09-30,500,1204.17,1883.36,679.19,0.0,9.8
2022-10-31,500,3320.67,3310.16,-10.52,0.0,10.0
2022-11-30,500,1244.25,1584.65,340.4,0.0,10.8


Random routing delta in euros · median +383.99 · 5th to 95th percentile +300.45 to +504.81
Classical-only common model: Last balance


group,routed_model
str,str
"""G01""","""Last balance"""
"""G02""","""SMA 28"""
"""G03""","""SMA 28"""


Account-origins,Common MAE (€),Routed MAE (€),Mean delta (€),Median delta (€),Helped (%)
u32,f64,f64,f64,f64,f64
1500,1978.56,2297.85,319.29,0.0,43.9


## Conclusion

**We choose Chronos-2 Small as the common Stage 3 benchmark and retain group-specific challengers.** In the run with both foundation models, we selected Chronos-2 Small on the first five origins for the common policy and observed **€1,923.03 MAE** on the last three. Routing selected Chronos-2 Small for G01 and G02 and SMA 28 for G03. We observed **€2,259.39 MAE**, 17.5% higher. The €336.36 increase fell within the range from random grouping. Classical-only routing also raised MAE from €1,978.56 to €2,297.85.

| Group | What we found | Stage 3 shortlist |
|---|---|---|
| **G01** | Lower activity and longer flat runs favor carrying the current level forward. Chronos-2 Small scored €1,124.49 and last balance scored €1,133.95. All three moving averages increased error. | We retain last balance and Chronos-2 Small and consider a revised calendar model that separates recurring inflows from irregular outflows. |
| **G02** | Frequent zero crossings coincide with the clearest benefit from smoothing. Last balance scored €393.46, SMA 14 scored €359.03, local-level smoothing scored €348.75 and Chronos-2 Small scored €274.84. | We shortlist Chronos-2 Small, local-level smoothing and SMA 14. |
| **G03** | We observed high activity and recurring inflow timing. The tested calendar models raised 30-day error, and SMA 28 scored €5,064.84. Chronos-2 Small scored €4,133.96 and last balance scored €4,168.20. | We shortlist Chronos-2 Small and last balance and retain a regularized calendar model for tests on revised features. |

We found persistent trends in roughly one third of account-origins across all groups. Damped Holt improved on last balance only in G02, where local-level smoothing performed better. Repeating the previous week's balances increased error in every group. We found evidence of calendar activity and a benefit from smoothing in G02. Routing by group raised average error in this pilot. For Stage 3, we proposed testing the shortlists and an additional foundation family on more windows, using Chronos-2 Small as the common benchmark and pairing errors by account. We evaluated three late-2022 windows with fixed model settings in this pilot.